In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/"
EXPORT_DIR = "/content/drive/MyDrive/HW4_MLOps_Export/"

os.makedirs(EXPORT_DIR, exist_ok=True)
print("Export folder ready:", EXPORT_DIR)

Export folder ready: /content/drive/MyDrive/HW4_MLOps_Export/


In [ ]:
files = {
    "orders_df": "olist_orders_dataset.csv.zip",
    "order_items_df": "olist_order_items_dataset.csv.zip",
    "order_payments_df": "olist_order_payments_dataset.csv.zip",
    "order_reviews_df": "olist_order_reviews_dataset.csv.zip",
    "customers_df": "olist_customers_dataset.csv.zip",
    "products_df": "olist_products_dataset.csv.zip",
    "sellers_df": "olist_sellers_dataset.csv",
    "category_translation_df": "product_category_name_translation.csv"
}

def load_any_csv(path):
    if path.endswith(".zip"):
        return pd.read_csv(path, compression="zip")
    return pd.read_csv(path)

tables = {name: load_any_csv(DATA_DIR + fname) for name, fname in files.items()}

orders_df = tables["orders_df"]
order_items_df = tables["order_items_df"]
order_payments_df = tables["order_payments_df"]
order_reviews_df = tables["order_reviews_df"]
customers_df = tables["customers_df"]
products_df = tables["products_df"]
sellers_df = tables["sellers_df"]
category_translation_df = tables["category_translation_df"]

print("All tables loaded successfully.")

All tables loaded successfully.


# Part 2 — Experiment Tracking with MLflow

In this section, I use MLflow to log multiple model runs for the Olist customer satisfaction problem. The goal is to track parameters, metrics, and artifacts so that results are reproducible and comparable.

For HW4, I log two runs:
1. A Random Forest model
2. A Gradient Boosting model

For each run, I record:
- model type and key hyperparameters
- accuracy, precision, recall, F1, and ROC-AUC
- the trained model artifact

In [ ]:
!pip install -q mlflow==2.19.0 joblib scikit-learn pandas numpy

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/"
MLFLOW_DIR = "/content/drive/MyDrive/HW4_MLflow"

os.makedirs(MLFLOW_DIR, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("MLFLOW_DIR:", MLFLOW_DIR)

DATA_DIR: /content/drive/MyDrive/Colab Notebooks/
MLFLOW_DIR: /content/drive/MyDrive/HW4_MLflow


## Load the Olist data and rebuild the deployable feature set

To keep the experiment tracking consistent with the deployed API, I rebuild the same structured Olist feature set used for the HW4 model.

In [ ]:
files = {
    "orders_df": "olist_orders_dataset.csv.zip",
    "order_items_df": "olist_order_items_dataset.csv.zip",
    "order_payments_df": "olist_order_payments_dataset.csv.zip",
    "order_reviews_df": "olist_order_reviews_dataset.csv.zip",
    "customers_df": "olist_customers_dataset.csv.zip",
    "products_df": "olist_products_dataset.csv.zip",
    "sellers_df": "olist_sellers_dataset.csv",
    "category_translation_df": "product_category_name_translation.csv"
}

def load_any_csv(path):
    if path.endswith(".zip"):
        return pd.read_csv(path, compression="zip")
    return pd.read_csv(path)

tables = {name: load_any_csv(DATA_DIR + fname) for name, fname in files.items()}

orders_df = tables["orders_df"]
order_items_df = tables["order_items_df"]
order_payments_df = tables["order_payments_df"]
order_reviews_df = tables["order_reviews_df"]
products_df = tables["products_df"]
sellers_df = tables["sellers_df"]
category_translation_df = tables["category_translation_df"]

print("Loaded source tables.")

Loaded source tables.


In [ ]:
products_en = products_df.copy()

if "product_category" not in products_en.columns:
    products_en = products_en.merge(
        category_translation_df,
        on="product_category_name",
        how="left"
    )
    products_en = products_en.rename(
        columns={"product_category_name_english": "product_category"}
    )

df = orders_df.merge(
    order_reviews_df[["order_id", "review_score"]],
    on="order_id",
    how="inner"
)

df = df[df["review_score"].notna()].copy()
df["is_positive_review"] = (df["review_score"] >= 4).astype(int)

order_items_agg = (
    order_items_df.groupby("order_id")
    .agg(
        price=("price", "sum"),
        freight_value=("freight_value", "sum"),
        product_id=("product_id", "first"),
        seller_id=("seller_id", "first")
    )
    .reset_index()
)

payments_agg = (
    order_payments_df.groupby("order_id")
    .agg(
        payment_type=("payment_type", "first")
    )
    .reset_index()
)

df = df.merge(order_items_agg, on="order_id", how="left")
df = df.merge(payments_agg, on="order_id", how="left")
df = df.merge(
    products_en[["product_id", "product_category"]],
    on="product_id",
    how="left"
)
df = df.merge(
    sellers_df[["seller_id", "seller_state"]],
    on="seller_id",
    how="left"
)

df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"], errors="coerce")
df["order_estimated_delivery_date"] = pd.to_datetime(df["order_estimated_delivery_date"], errors="coerce")

df["delivery_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

df["delivery_vs_estimated"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

feature_cols = [
    "delivery_days",
    "delivery_vs_estimated",
    "price",
    "freight_value",
    "product_category",
    "seller_state",
    "payment_type"
]

target_col = "is_positive_review"

df_model = df[feature_cols + [target_col]].dropna().copy()
print("Modeling shape:", df_model.shape)
df_model.head()

Modeling shape: (94981, 8)


,delivery_days,delivery_vs_estimated,price,freight_value,product_category,seller_state,payment_type,is_positive_review
0,8.0,-8.0,29.99,8.72,housewares,SP,credit_card,1
1,13.0,-6.0,118.70,22.76,perfumery,SP,boleto,1
2,9.0,-18.0,159.90,19.22,auto,SP,credit_card,1
3,13.0,-13.0,45.00,27.20,pet_shop,MG,credit_card,1
4,2.0,-10.0,19.90,8.72,stationery,SP,credit_card,1


In [ ]:
X = df_model[feature_cols].copy()
y = df_model[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ["delivery_days", "delivery_vs_estimated", "price", "freight_value"]
categorical_features = ["product_category", "seller_state", "payment_type"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (75984, 7)
Test shape: (18997, 7)


## Set up MLflow

The HW4 instructions require an experiment named `olist-satisfaction`. I store the MLflow tracking files in Google Drive so the outputs persist across sessions.

In [ ]:
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR}")
mlflow.set_experiment("olist-satisfaction")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment set to: olist-satisfaction")

Tracking URI: file:///content/drive/MyDrive/HW4_MLflow
Experiment set to: olist-satisfaction


In [ ]:
def evaluate_model(model_pipeline, X_test, y_test):
    y_pred = model_pipeline.predict(X_test)
    y_proba = model_pipeline.predict_proba(X_test)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }

## Run 1 — Random Forest

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    ))
])

with mlflow.start_run(run_name="random_forest_run"):
    rf_pipeline.fit(X_train, y_train)
    rf_metrics = evaluate_model(rf_pipeline, X_test, y_test)

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 20)
    mlflow.log_param("random_state", 42)

    for metric_name, metric_value in rf_metrics.items():
        mlflow.log_metric(metric_name, metric_value)

    mlflow.sklearn.log_model(
        sk_model=rf_pipeline,
        artifact_path="model",
        registered_model_name="olist-satisfaction-model"
    )

rf_metrics

2026/04/17 18:58:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'olist-satisfaction-model' already exists. Creating a new version of this model...
Created version '3' of model 'olist-satisfaction-model'.


{'accuracy': 0.8228667684371217,
 'precision': 0.8239126802918082,
 'recall': 0.9864648619815976,
 'f1': 0.8978910635715369,
 'roc_auc': np.float64(0.6869918989948848)}

## Run 2 — Gradient Boosting

In [ ]:
gb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.1,
        random_state=42
    ))
])

with mlflow.start_run(run_name="gradient_boosting_run"):
    gb_pipeline.fit(X_train, y_train)
    gb_metrics = evaluate_model(gb_pipeline, X_test, y_test)

    mlflow.log_param("model_type", "GradientBoostingClassifier")
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("random_state", 42)

    for metric_name, metric_value in gb_metrics.items():
        mlflow.log_metric(metric_name, metric_value)

    mlflow.sklearn.log_model(
        sk_model=gb_pipeline,
        artifact_path="model",
        registered_model_name="olist-satisfaction-model"
    )

gb_metrics

2026/04/17 18:58:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'olist-satisfaction-model' already exists. Creating a new version of this model...
Created version '4' of model 'olist-satisfaction-model'.


{'accuracy': 0.8230246881086487,
 'precision': 0.8236177550339303,
 'recall': 0.9872649686624884,
 'f1': 0.8980470645317807,
 'roc_auc': np.float64(0.6849909903799459)}

## Compare the two runs

In [ ]:
mlflow_summary = pd.DataFrame([
    {
        "Run": "Random Forest",
        "Accuracy": rf_metrics["accuracy"],
        "Precision": rf_metrics["precision"],
        "Recall": rf_metrics["recall"],
        "F1": rf_metrics["f1"],
        "ROC_AUC": rf_metrics["roc_auc"]
    },
    {
        "Run": "Gradient Boosting",
        "Accuracy": gb_metrics["accuracy"],
        "Precision": gb_metrics["precision"],
        "Recall": gb_metrics["recall"],
        "F1": gb_metrics["f1"],
        "ROC_AUC": gb_metrics["roc_auc"]
    }
])

mlflow_summary

,Run,Accuracy,Precision,Recall,F1,ROC_AUC
0,Random Forest,0.822867,0.823913,0.986465,0.897891,0.686992
1,Gradient Boosting,0.823025,0.823618,0.987265,0.898047,0.684991


## Determine the best run

In [ ]:
best_model_name = (
    "Random Forest"
    if rf_metrics["roc_auc"] >= gb_metrics["roc_auc"]
    else "Gradient Boosting"
)

print("Best model based on ROC-AUC:", best_model_name)

Best model based on ROC-AUC: Random Forest


## Next step outside the notebook

After running these cells, start the MLflow UI locally and take screenshots of:
1. the Experiments page showing both runs
2. the Model Registry page showing the registered model

Then move the best model version to the **Production** stage, as required by HW4.

### MLflow notes for HW4

This section satisfies the core tracking requirement because it:
- creates the `olist-satisfaction` experiment
- logs at least two runs
- records parameters and metrics
- logs the trained model artifact
- registers the model in the MLflow Model Registry